# Player-Race MMR by Date

Load the daily player-race MMR aggregation and plot average MMR trends for the most active player-race combinations.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from analysis import load_replays, build_player_race_mmr_by_day, get_latest_player_race_mmrs
from scipy.stats import logistic

replays = load_replays()
rows = build_player_race_mmr_by_day(replays)
df = pd.DataFrame(rows)
df['day'] = pd.to_datetime(df['day'])
df = df.sort_values(['player_name', 'race', 'day'])

df = df.loc[(df['race'] == 'Z') & (df['avg_mmr'] >= 2500)]

player_game_counts = df.groupby('player_name')['replay_count'].sum()
active_players = player_game_counts[player_game_counts > 10].index

df = df[df['player_name'].isin(active_players)]

print(f"Number of active players: {df['player_name'].size}")

df.head()


Number of active players: 3168


,player_name,race,day,replay_count,avg_mmr,min_mmr,max_mmr,wins,losses
2156,AdobeKenobi,Z,2026-06-10,2,2723.500000,2723.500000,2723.500000,1,1
2157,AdobeKenobi,Z,2026-06-11,7,2730.142857,2730.142857,2730.142857,4,3
2158,AdobeKenobi,Z,2026-06-12,10,2816.500000,2816.500000,2816.500000,8,2
2159,AdobeKenobi,Z,2026-06-13,17,2964.941176,2964.941176,2964.941176,9,8
2160,AdobeKenobi,Z,2026-06-14,5,2830.400000,2830.400000,2830.400000,2,3


In [2]:
import plotly.express as px

# Plot each Z player as a separate line with markers and hover labels
fig = px.line(
    df,
    x='day',
    y='avg_mmr',
    color='player_name',
    line_group='player_name',
    markers=True,
    hover_data=['race', 'replay_count', 'min_mmr', 'max_mmr', 'wins', 'losses'],
    title='Player-Race Daily Average MMR (Z only)',
)
fig.update_traces(opacity=0.35)

# Add a thick group average line on top
group_avg = df.groupby('day', as_index=False)['avg_mmr'].mean()
fig.add_scatter(
    x=group_avg['day'],
    y=group_avg['avg_mmr'],
    mode='lines',
    line={'color': 'black', 'width': 4},
    name='Group average',
    hovertemplate='day=%{x}<br>avg_mmr=%{y:.1f}<extra></extra>',
)

fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Average MMR',
    legend_title='Player',
    hovermode='closest',
    template='plotly_white',
    legend=dict(itemclick='toggleothers', itemdoubleclick='toggle'),
    height=1200,
    width=1600,
)

fig.show()


In [5]:
replays = load_replays()
mmrs = get_latest_player_race_mmrs(replays)
all_mmrs = [m for vals in mmrs.values() for m in vals]
if not all_mmrs:
    print('No MMR data to plot')
else:
    bin_size = 100
    start = (min(all_mmrs) // bin_size) * bin_size
    end = ((max(all_mmrs) // bin_size) + 1) * bin_size
    bins = np.arange(start, end + bin_size, bin_size)
    labels = [f"{b}-{b+bin_size-1}" for b in bins[:-1]]
    centers = (bins[:-1] + bins[1:]) / 2

    counts = {}
    for race in ['P','Z','T']:
        counts[race], _ = np.histogram(mmrs[race], bins=bins)

    fig = go.Figure()
    colors = {'P':'green','Z':'red','T':'blue'}
    bin_width = bins[1] - bins[0]

    for race in ['P','Z','T']:
        n = int(counts[race].sum())
        fig.add_trace(go.Scatter(
            x=centers,
            y=counts[race],
            mode='lines+markers',
            name=f"{race} (n={n})",
            line=dict(color=colors[race]),
            marker=dict(size=6),
            hovertemplate='bin_center=%{x:.0f}<br>count=%{y}<extra></extra>'
        ))

        # overlay logistic fit (scaled to counts per bin)
        if n > 0:
            vals = np.array(mmrs[race])
            try:
                loc, scale = logistic.fit(vals)
                pdf = logistic.pdf(centers, loc=loc, scale=scale)
                pdf_counts = pdf * n * bin_width
                # compute percentiles (0-100) from the logistic CDF for hover info
                percentiles = logistic.cdf(centers, loc=loc, scale=scale) * 100
                fig.add_trace(go.Scatter(
                    x=centers,
                    y=pdf_counts,
                    mode='lines',
                    name=f"{race} logistic fit",
                    line=dict(color=colors[race], dash='dash'),
                    customdata=percentiles,
                    hovertemplate='mmr=%{x:.0f}<br>fit_count=%{y:.1f}<br>percentile=%{customdata:.1f}%<extra></extra>'
                ))
            except Exception as e:
                print(f'Logistic fit failed for {race}: {e}')

    fig.update_layout(
        title='MMR Distribution by Race (frequency polygon)',
        xaxis_title='MMR bin center',
        yaxis_title='Count',
        template='plotly_white',
        width=1600,
        height=800,
    )
    fig.update_xaxes(tickvals=centers, ticktext=labels, tickangle=-45)
    fig.show()

In [4]:
replays = load_replays()
mmrs = get_latest_player_race_mmrs(replays)
counts = {race: len(vals) for race, vals in mmrs.items()}
total_unique = sum(counts.values())
print('Counts per race:', counts)
print('Total unique player-race samples:', total_unique)

stats_rows = []
for race, vals in mmrs.items():
    arr = np.array(vals)
    n = len(arr)
    mean = float(arr.mean()) if n else None
    median = float(np.median(arr)) if n else None
    std = float(arr.std(ddof=0)) if n else None
    stats_rows.append({'race': race, 'n': n, 'mean': mean, 'median': median, 'std': std})
stats_df = pd.DataFrame(stats_rows).set_index('race')
print()
display(stats_df)
stats_df.to_csv('mmr_stats_by_race.csv')
print('Saved mmr_stats_by_race.csv')


Counts per race: {'P': 2776, 'T': 3193, 'Z': 3553}
Total unique player-race samples: 9522



,n,mean,median,std
race,,,,
P,2776,3312.449928,3274.5,747.218216
T,3193,3208.490761,3117.0,742.662048
Z,3553,3379.645089,3300.0,717.364255


Saved mmr_stats_by_race.csv
